<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c07-header.png" alt="Nextia Learning · LLMs: How They Work and How to Choose One" width="100%">

# Predicting continuations

**[Predicting continuations](https://learning.nextia-ai.com/courses/llms/m01/predicting-continuations/)** · LLMs: How They Work and How to Choose One · Module 1, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn a model's log-probabilities into probabilities, re-weight them with the temperature, and sample the next token, on real numbers from Gemma 3 4B.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | An internet connection for the setup cell. No account, no API key and no model: the notebook uses probabilities recorded on 2026-10-08. |
| **You should know** | How to run a notebook cell, and Python lists and functions (from the Python course). A probability and a distribution (from the mathematics course). |
| **Optional** | The lesson makes complete sense without this notebook. |
| **Tested** | 2026-10-08, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/llms/m01/predicting-continuations/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C07/M01-L02-predicting-continuations/predicting-continuations-solution.ipynb).

## Setup

The next cell installs nothing, downloads the course data from the [labs repository](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C07/data) and checks each file. It also defines `expect()` and `expect_hash()`, which check your answers without showing them.

> **Check.** The last line starts with `Ready:`. If you see a checksum error, delete the folder `c07-data` and run the cell again.

In [ ]:
# Setup: packages and data. Run this cell first.
import hashlib, json, subprocess, sys, urllib.request
from pathlib import Path
# No packages to install.

DATA_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C07/data/"
FILES = {'m01-next-token.json': '5d1e335da85dae24b0450e501681bec2dd1b1cf0d3d271dde90c606c95d825ad'}
DATA = Path("c07-data")
DATA.mkdir(exist_ok=True)
for name, expected in FILES.items():
    path = DATA / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        try:
            urllib.request.urlretrieve(DATA_URL + name, path)
        except OSError:
            subprocess.run(["curl", "-fsSL", "-o", str(path), DATA_URL + name], check=True)
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder c07-data and run again.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare your number with the expected one, without showing the answer."""
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tolerance:
            print(f"Check passed: {what} is right.")
        else:
            print(f"Not yet: {what} is too {'high' if yours > expected else 'low'}.")
    elif yours == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected value.")

def fingerprint(value):
    """A short hash of a value (numbers rounded to 2 decimals)."""
    norm = round(value, 2) if isinstance(value, float) else value
    return hashlib.sha256(repr(norm).encode()).hexdigest()[:12]

def expect_hash(what, yours, expected_hash):
    """Compare your answer with a hidden fingerprint of the expected one."""
    if fingerprint(yours) == expected_hash:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected value. Check the steps above.")

print("Ready:", ", ".join(FILES))

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The recorded probabilities

On 2026-10-08, Gemma 3 4B (through Ollama 0.40.1) gave its 20 most likely next tokens for the text "The parcel arrived late, so the customer asked for a". Ollama returns **log-probabilities**: the natural logarithm of each probability. `math.exp` turns one back into a probability.

> **Check.** The first line is `' refund' 0.913`, and the 20 probabilities add up to about 0.995.

In [ ]:
import math, random

recorded = json.loads((DATA / "m01-next-token.json").read_text(encoding="utf-8"))
top = recorded["next"]["en"]["top"]          # 20 tokens: {"t": token, "lp": log-probability}
tokens = [x["t"] for x in top]
logprobs = [x["lp"] for x in top]
for t, lp in list(zip(tokens, logprobs))[:5]:
    print(f"{t!r:12} {math.exp(lp):.3f}")
print("sum of the 20:", round(sum(math.exp(lp) for lp in logprobs), 3))

## 2. Your turn: the temperature

The temperature T divides every log-probability by T; then the shares are scaled so that they add up to 1 (a softmax). This is the same as raising each probability to the power 1/T.

> **Predict.** At temperature 2, does the share of `' refund'` go up or down?

> **Your turn.** Complete `at_temperature(logprobs, T)`. Return a list of probabilities that add up to 1.

In [ ]:
def at_temperature(logprobs, T):
    # your code here: exp(lp / T) for each token, then divide each by their sum
    return None

shares = at_temperature(logprobs, 2.0)
print(shares and [round(p, 3) for p in shares[:5]])

Run the check. It compares the share of `' refund'` at temperature 2 and at 0.5 with the explorer of the lesson.

In [ ]:
expect_hash("the share of ' refund' at temperature 2", round(at_temperature(logprobs, 2.0)[0], 2), "7fb14de5048d")
expect_hash("the share of ' refund' at temperature 0.5", round(at_temperature(logprobs, 0.5)[0], 2), "d0ff5974b6aa")

## 3. Sampling with a seed

Sampling draws one token at random, with the shares as chances. `random.Random(seed)` makes the draws repeatable: the same seed gives the same draws.

> **Predict.** In 200 draws at temperature 2, about how many are `' refund'`?

In [ ]:
def sample(shares, n, seed):
    rng = random.Random(seed)
    draws = rng.choices(tokens, weights=shares, k=n)
    return {t: draws.count(t) for t in tokens[:5]}

print("temperature 1:", sample(at_temperature(logprobs, 1.0), 200, seed=7))
print("temperature 2:", sample(at_temperature(logprobs, 2.0), 200, seed=7))

Compare with what the real sampler did. Amira drew 200 one-token samples from Gemma 3 at each temperature (seeds 1 to 200) and counted them:

In [ ]:
for run in recorded["check"]:
    print(f"temperature {run['temperature']}:")
    for row in run["rows"][:5]:
        print(f"  {row['t']!r:12} expected {row['expected']:6.1f}   counted {row['observed']}")

> **Check.** Your draws, the formula and the real counts are close, but not equal. With 200 draws, a difference of a few counts is normal.

## 4. Change one thing: the French sentence

> **Your turn.** Use `recorded["next"]["fr"]["top"]` to print the share of the top French token at temperatures 1, 1.5 and 2. Is the model more or less sure than in English?

In [ ]:
fr_logprobs = [x["lp"] for x in recorded["next"]["fr"]["top"]]
# your code here

> **Check.** ` remboursement` has about 0.994 at temperature 1 and about 0.789 at 2: the model is more sure in French than in English here.

## 5. Failure case: one draw is not the distribution

Run the cell: five draws of the first token, each with another seed, at temperature 1.

In [ ]:
for seed in range(5):
    print(seed, random.Random(seed).choices(tokens, weights=at_temperature(logprobs, 1.0))[0])

> **Check.** Most draws are ` refund`, but not always. A single reply that you read is one draw. Judge a model on many runs.

## 6. Optional: ask a local model yourself

This cell works only if you run Ollama on your own computer with `gemma3:4b` (it does not work in Colab or Kaggle). It is not needed for the lesson.

In [ ]:
# Optional: needs Ollama running locally with gemma3:4b.
import urllib.request
body = {"model": "gemma3:4b", "prompt": recorded["prompts"]["en"], "raw": True, "stream": False,
        "logprobs": True, "top_logprobs": 5, "options": {"temperature": 0, "num_predict": 1}}
try:
    req = urllib.request.Request("http://localhost:11434/api/generate", json.dumps(body).encode(), {"Content-Type": "application/json"})
    answer = json.load(urllib.request.urlopen(req, timeout=120))
    for item in answer["logprobs"][0]["top_logprobs"]:
        print(f"{item['token']!r:12} {math.exp(item['logprob']):.3f}")
except OSError as error:
    print("No local Ollama here; that is fine. The recorded numbers above are the same run.", error)

## Recap

- A model gives every token a probability; a log-probability turns into one with `math.exp`.
- The temperature re-weights the probabilities: below 1 the top token gains, above 1 the shares become more even.
- Sampling with a seed is repeatable; one draw is not the whole distribution.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Training stages](https://learning.nextia-ai.com/courses/llms/m01/training-stages/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/llms/m01/predicting-continuations/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.